# Translation Style Fine-Tuning

**Goal:** fine-tune a language model to produce domain-specific translations that match a particular style, register, and terminology — then evaluate whether the fine-tuning actually improved on the base model.

---

## Why Style Matters in Translation

Generic translation models produce correct but generic output. Domain-specific translation requires:

| Dimension | Generic model output | Domain-tuned output |
|---|---|---|
| **Terminology** | "income statement" | "profit and loss account" (UK accounting) |
| **Register** | formal/neutral mix | consistently formal (legal) or casual (marketing) |
| **Sentence structure** | literal, source-language calque | natural target-language phrasing for the domain |
| **Abbreviations** | spelled out | domain-standard abbreviations used correctly |
| **Tone** | neutral | matches brand or institutional voice |

## What This Notebook Covers

1. Define the domain, style policy, and terminology constraints
2. Curate a parallel dataset with style-aware translations
3. Format examples for supervised fine-tuning
4. Design train/validation splits that avoid leakage
5. Configure LoRA fine-tuning
6. Evaluate with translation metrics and style-specific checks
7. Discuss limitations and failure modes

## Pipeline

```
Source text (English, domain-specific)
        │
        ▼
┌───────────────────────┐
│  Fine-tuned model     │  Translate with domain style
└──────────┬────────────┘
           │
           ▼
┌───────────────────────┐
│  Terminology check    │  Required terms used correctly?
└──────────┬────────────┘
           │
           ▼
┌───────────────────────┐
│  Style rubric eval    │  Register, tone, structure match?
└──────────┬────────────┘
           │
           ▼
   Domain-appropriate translation
```

## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy scikit-learn datasets transformers peft trl accelerate evaluate sacrebleu rouge-score

In [ ]:
import json
import random
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RUN_TRAINING = False
RUN_GENERATION_EVAL = False
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = ARTIFACT_DIR / "translation-style-lora"

print(f"Project dir:  {PROJECT_DIR}")
print(f"Artifacts:    {ARTIFACT_DIR}")
print(f"Base model:   {BASE_MODEL}")
print(f"Training:     {RUN_TRAINING}")
print(f"Gen eval:     {RUN_GENERATION_EVAL}")

## 2. Define the Domain and Style Policy

### Choosing a Domain: Financial Regulatory Translation (EN → FR)

We pick **financial regulatory documents** translated from English to French. This domain has:

- a **fixed terminology** defined by the EU and French financial regulators
- a **formal register** required in regulatory filings
- **structural conventions** (numbered clauses, defined terms in quotes, passive voice)
- real-world demand — mistranslation in regulatory filings has legal consequences

### Style Policy

The style policy is the single source of truth that the model must learn.

| Rule | Example (wrong → right) |
|---|---|
| Use official EU/FR financial terminology | "actions" → "titres de capital" for equity shares |
| Keep defined terms from the source in quotes on first use | Art. 12 "Eligible Counterparty" → Art. 12 « Contrepartie éligible » |
| Maintain formal register throughout | "on doit" → "il convient de" |
| Preserve numbered clause structure | (a), (b), (c) → (a), (b), (c) — do not renumber |
| Use the institutional third person | "we require" → "l'Autorité exige" |
| French typographic conventions for punctuation | space before : ; ? ! and use « » not " " |

### Why This Cannot Be Solved by Prompting Alone

- Terminology consistency across a 50-page document drifts with prompting
- Register and institutional tone require hundreds of calibration examples
- Punctuation conventions are fiddly and forgotten after a few paragraphs
- The same English term may need different French translations in different regulatory contexts

In [ ]:
STYLE_POLICY = {
    "source_language": "English",
    "target_language": "French",
    "domain": "Financial regulatory (EU/French law)",
    "register": "Formal institutional",
    "terminology_rules": [
        "Use official EU financial terminology as defined in MiFID II / MiFIR French versions",
        "Translate 'eligible counterparty' as 'contrepartie eligible'",
        "Translate 'investment firm' as 'entreprise d'investissement'",
        "Translate 'competent authority' as 'autorite competente'",
        "Translate 'financial instrument' as 'instrument financier'",
        "Translate 'regulated market' as 'marche reglemente'",
        "Translate 'best execution' as 'meilleure execution'",
        "Translate 'systematic internaliser' as 'internalisateur systematique'",
    ],
    "style_rules": [
        "Use formal register consistently (il convient de, not on doit)",
        "Use institutional third person (l'Autorite, not nous)",
        "Preserve numbered clause structure from source",
        "Use French typographic conventions (space before : ; ? !)",
        "Use guillemets for quotes and defined terms",
        "Maintain passive constructions where source uses passive",
    ],
}

print(json.dumps(STYLE_POLICY, indent=2, ensure_ascii=False))

## 3. Dataset Curation

### What Makes a Good Translation Fine-Tuning Dataset?

| Quality dimension | Why it matters | How to check |
|---|---|---|
| **Parallel alignment** | Source and target must be semantically equivalent | Human bilingual review |
| **Terminology consistency** | Same English term → same French term throughout | Automated glossary check |
| **Style consistency** | Same register, tone, and conventions in every example | Style rubric scoring |
| **Domain coverage** | Examples from all relevant sub-topics | Topic distribution analysis |
| **Edge cases** | Abbreviations, cross-references, nested clauses | Slice-based testing |

### Curation Workflow

1. Collect existing official translations (EU regulatory documents often have parallel EN/FR versions)
2. Filter to the target domain and style register
3. Segment into sentence or clause-level pairs
4. Review and correct terminology alignment against the glossary
5. Tag by sub-topic, complexity, and source document
6. Validate every pair against the style policy

### Sources for Real Projects

- EUR-Lex parallel corpus (EU legislation in all official languages)
- National financial regulator publications (AMF, ESMA)
- Company annual reports with official FR translations
- Professional translator output from translation memory systems

In [ ]:
# Synthetic parallel dataset for this demo.
# Each record has: source (EN), target (FR), sub-topic, complexity, and notes.
pairs = [
    {
        "pair_id": "P-001",
        "source": "The investment firm shall ensure best execution of client orders at all times.",
        "target": "L'entreprise d'investissement veille a assurer la meilleure execution des ordres de ses clients en toute circonstance.",
        "topic": "best_execution",
        "complexity": "simple",
    },
    {
        "pair_id": "P-002",
        "source": "Eligible counterparties are not afforded the same level of protection as retail clients.",
        "target": "Les contreparties eligibles ne beneficient pas du meme niveau de protection que les clients de detail.",
        "topic": "client_classification",
        "complexity": "simple",
    },
    {
        "pair_id": "P-003",
        "source": "The competent authority may, at any time, require the investment firm to demonstrate that its execution policy is adequate.",
        "target": "L'autorite competente peut, a tout moment, exiger de l'entreprise d'investissement qu'elle demontre le caractere adequat de sa politique d'execution.",
        "topic": "best_execution",
        "complexity": "medium",
    },
    {
        "pair_id": "P-004",
        "source": "Financial instruments admitted to trading on a regulated market must comply with the transparency requirements set out in Title III.",
        "target": "Les instruments financiers admis a la negociation sur un marche reglemente doivent satisfaire aux exigences de transparence enoncees au titre III.",
        "topic": "transparency",
        "complexity": "medium",
    },
    {
        "pair_id": "P-005",
        "source": "A systematic internaliser shall make public firm quotes in respect of those shares for which it is a systematic internaliser and for which there is a liquid market.",
        "target": "Un internalisateur systematique publie des prix fermes pour les actions pour lesquelles il est internalisateur systematique et pour lesquelles il existe un marche liquide.",
        "topic": "systematic_internaliser",
        "complexity": "complex",
    },
    {
        "pair_id": "P-006",
        "source": "Member States shall require that investment firms authorised under this Directive satisfy the conditions laid down in Article 14.",
        "target": "Les Etats membres exigent que les entreprises d'investissement agreees en vertu de la presente directive remplissent les conditions prevues a l'article 14.",
        "topic": "authorisation",
        "complexity": "medium",
    },
    {
        "pair_id": "P-007",
        "source": "The order execution policy shall include, for each class of financial instruments, information on the venues where the firm executes its client orders.",
        "target": "La politique d'execution des ordres comprend, pour chaque categorie d'instruments financiers, des informations sur les plates-formes sur lesquelles l'entreprise execute les ordres de ses clients.",
        "topic": "best_execution",
        "complexity": "complex",
    },
    {
        "pair_id": "P-008",
        "source": "Without prejudice to the provisions of this Directive, the competent authority shall have the power to suspend trading.",
        "target": "Sans prejudice des dispositions de la presente directive, l'autorite competente a le pouvoir de suspendre la negociation.",
        "topic": "market_operations",
        "complexity": "medium",
    },
    {
        "pair_id": "P-009",
        "source": "The provisions of this Regulation shall apply to credit institutions when providing investment services and activities.",
        "target": "Les dispositions du present reglement s'appliquent aux etablissements de credit lorsqu'ils fournissent des services et activites d'investissement.",
        "topic": "scope",
        "complexity": "simple",
    },
    {
        "pair_id": "P-010",
        "source": "An investment firm shall not be regarded as dealing on own account when it executes client orders.",
        "target": "Une entreprise d'investissement n'est pas consideree comme negociant pour compte propre lorsqu'elle execute des ordres de clients.",
        "topic": "definitions",
        "complexity": "medium",
    },
    {
        "pair_id": "P-011",
        "source": "The firm shall establish and implement effective organisational and administrative arrangements to prevent conflicts of interest from adversely affecting client interests.",
        "target": "L'entreprise etablit et met en oeuvre des dispositions organisationnelles et administratives efficaces pour empecher que les conflits d'interets ne portent atteinte aux interets des clients.",
        "topic": "conflicts_of_interest",
        "complexity": "complex",
    },
    {
        "pair_id": "P-012",
        "source": "Where the firm is unable to prevent a conflict of interest, it shall clearly disclose the nature and source of the conflict to the client before undertaking business on behalf of the client.",
        "target": "Lorsque l'entreprise n'est pas en mesure de prevenir un conflit d'interets, elle revele clairement au client la nature et la source du conflit avant d'agir pour le compte de ce client.",
        "topic": "conflicts_of_interest",
        "complexity": "complex",
    },
    {
        "pair_id": "P-013",
        "source": "ESMA shall develop draft regulatory technical standards specifying the criteria for determining whether an activity is to be considered as ancillary to the main business.",
        "target": "L'AEMF elabore des projets de normes techniques de reglementation precisant les criteres permettant de determiner si une activite doit etre consideree comme auxiliaire par rapport a l'activite principale.",
        "topic": "scope",
        "complexity": "complex",
    },
    {
        "pair_id": "P-014",
        "source": "Pursuant to Article 5(1), the competent authority shall grant authorisation only where it is satisfied that the applicant meets the requirements.",
        "target": "Conformement a l'article 5, paragraphe 1, l'autorite competente n'accorde l'agrement que si elle est convaincue que le demandeur satisfait aux exigences.",
        "topic": "authorisation",
        "complexity": "medium",
    },
    {
        "pair_id": "P-015",
        "source": "Information provided to clients shall be fair, clear and not misleading.",
        "target": "Les informations communiquees aux clients sont correctes, claires et non trompeuses.",
        "topic": "client_information",
        "complexity": "simple",
    },
    {
        "pair_id": "P-016",
        "source": "The management body of the investment firm shall define, approve and oversee a policy as regards the provision of services and activities.",
        "target": "L'organe de direction de l'entreprise d'investissement definit, approuve et supervise une politique relative a la fourniture de services et d'activites.",
        "topic": "governance",
        "complexity": "medium",
    },
    {
        "pair_id": "P-017",
        "source": "Client financial instruments held by the firm shall be adequately safeguarded in accordance with the applicable national law.",
        "target": "Les instruments financiers des clients detenus par l'entreprise sont proteges de maniere adequate conformement au droit national applicable.",
        "topic": "client_assets",
        "complexity": "medium",
    },
    {
        "pair_id": "P-018",
        "source": "Trading venues shall make pre-trade and post-trade data available on a reasonable commercial basis.",
        "target": "Les plates-formes de negociation mettent a disposition les donnees pre- et post-negociation a des conditions commerciales raisonnables.",
        "topic": "transparency",
        "complexity": "medium",
    },
    {
        "pair_id": "P-019",
        "source": "The delegated acts referred to in paragraph 2 shall be adopted in accordance with the procedure set out in Article 64.",
        "target": "Les actes delegues vises au paragraphe 2 sont adoptes conformement a la procedure prevue a l'article 64.",
        "topic": "procedural",
        "complexity": "simple",
    },
    {
        "pair_id": "P-020",
        "source": "Where the investment firm outsources critical or important operational functions, it shall remain fully responsible for discharging all of its obligations under this Directive.",
        "target": "Lorsque l'entreprise d'investissement externalise des fonctions operationnelles essentielles ou importantes, elle reste pleinement responsable du respect de toutes ses obligations au titre de la presente directive.",
        "topic": "outsourcing",
        "complexity": "complex",
    },
]

df = pd.DataFrame(pairs)
print(f"Parallel pairs: {len(df)}")
print(f"\nTopic distribution:")
print(df["topic"].value_counts().to_string())
print(f"\nComplexity distribution:")
print(df["complexity"].value_counts().to_string())

## 4. Dataset Quality Checks

Before training, validate the dataset against the style policy. Bad training data is the most common cause of bad translation style after fine-tuning.

### Automated Checks

- terminology compliance: required terms appear in targets
- length ratio: source/target length should be within expected range for EN→FR (FR is typically 10-20% longer)
- typographic conventions: French spacing rules, guillemets
- structural preservation: numbered clauses, article references

In [ ]:
# Terminology glossary for automated checking
GLOSSARY = {
    "investment firm": "entreprise d'investissement",
    "competent authority": "autorite competente",
    "eligible counterpart": "contrepartie eligible",
    "financial instrument": "instrument financier",
    "regulated market": "marche reglemente",
    "best execution": "meilleure execution",
    "systematic internaliser": "internalisateur systematique",
}


def check_terminology(source, target):
    issues = []
    source_lower = source.lower()
    target_lower = target.lower()
    for en_term, fr_term in GLOSSARY.items():
        if en_term in source_lower:
            # Allow for accented and unaccented versions
            fr_normalized = fr_term.replace("e'", "e").replace("e`", "e")
            target_normalized = target_lower.replace("\u00e9", "e").replace("\u00e8", "e")
            if fr_normalized not in target_normalized and fr_term not in target_lower:
                issues.append(f"Missing '{fr_term}' for '{en_term}'")
    return issues


def check_length_ratio(source, target, min_ratio=0.9, max_ratio=1.5):
    ratio = len(target) / max(len(source), 1)
    if ratio < min_ratio or ratio > max_ratio:
        return f"Length ratio {ratio:.2f} outside [{min_ratio}, {max_ratio}]"
    return None


def run_quality_checks(frame):
    results = []
    for _, row in frame.iterrows():
        issues = []
        term_issues = check_terminology(row["source"], row["target"])
        issues.extend(term_issues)
        length_issue = check_length_ratio(row["source"], row["target"])
        if length_issue:
            issues.append(length_issue)
        results.append({
            "pair_id": row["pair_id"],
            "issues": issues,
            "issue_count": len(issues),
        })
    return pd.DataFrame(results)


quality_report = run_quality_checks(df)
total_issues = quality_report["issue_count"].sum()
clean = (quality_report["issue_count"] == 0).sum()
print(f"Quality check: {clean}/{len(df)} pairs clean, {total_issues} total issues")
if total_issues > 0:
    flagged = quality_report[quality_report["issue_count"] > 0]
    for _, row in flagged.iterrows():
        print(f"  {row['pair_id']}: {'; '.join(row['issues'])}")

## 5. Format Examples for SFT

### System Prompt Design for Translation Style

The system prompt must encode three things:

1. **The translation direction** (EN → FR)
2. **The domain** (financial regulatory)
3. **Key style rules** (terminology, register, conventions)

Unlike general translation, we embed the glossary in the system prompt so the model has the terminology reference available during both training and inference.

In [ ]:
TRANSLATION_SYSTEM_PROMPT = (
    "You are a professional financial regulatory translator (English to French).\n"
    "Translate the following text into French, strictly adhering to these rules:\n\n"
    "TERMINOLOGY (use these exact terms):\n"
    "- investment firm = entreprise d'investissement\n"
    "- competent authority = autorite competente\n"
    "- eligible counterparty = contrepartie eligible\n"
    "- financial instrument = instrument financier\n"
    "- regulated market = marche reglemente\n"
    "- best execution = meilleure execution\n"
    "- systematic internaliser = internalisateur systematique\n\n"
    "STYLE RULES:\n"
    "- Use formal institutional register throughout\n"
    "- Use institutional third person (l'Autorite, l'entreprise), not first person\n"
    "- Preserve numbered clause structure, article references, and cross-references exactly\n"
    "- Use French typographic conventions (space before colon, semicolon, question and exclamation marks)\n"
    "- Output ONLY the French translation. No commentary, no explanation."
)


def to_chat_record(row):
    return {
        "pair_id": row["pair_id"],
        "topic": row["topic"],
        "complexity": row["complexity"],
        "messages": [
            {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
            {"role": "user", "content": row["source"]},
            {"role": "assistant", "content": row["target"]},
        ],
    }


chat_records = [to_chat_record(row) for _, row in df.iterrows()]
chat_df = pd.DataFrame(chat_records)

print(f"Formatted {len(chat_records)} chat records")
print(f"System prompt: {len(TRANSLATION_SYSTEM_PROMPT)} chars")
print(f"\nExample:")
print(json.dumps(chat_records[0]['messages'], indent=2, ensure_ascii=False))

In [ ]:
# Save JSONL for reuse
jsonl_path = ARTIFACT_DIR / "translation_style_sft.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for rec in chat_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"Saved: {jsonl_path}")

## 6. Train / Validation Split

### Split Strategy for Translation

| Leakage risk | Why it matters | Fix |
|---|---|---|
| Same document in both sets | Model memorizes phrasing, not translation skill | Split by source document |
| Same clause paraphrased in both | Near-duplicates inflate scores | Deduplicate before splitting |
| Terminology only in train | Eval does not test terminology compliance | Stratify by topic to cover all terms |
| All complex examples in train | Eval set is misleadingly easy | Stratify by complexity |

For this demo we stratify by complexity to ensure the validation set has a representative mix.

In [ ]:
train_df, val_df = train_test_split(
    chat_df,
    test_size=0.25,
    random_state=SEED,
    stratify=chat_df["complexity"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} pairs")
print(f"Val:   {len(val_df)} pairs")
print(f"\nTrain complexity: {dict(Counter(train_df['complexity']))}")
print(f"Val complexity:   {dict(Counter(val_df['complexity']))}")
print(f"\nTrain topics: {dict(Counter(train_df['topic']))}")
print(f"Val topics:   {dict(Counter(val_df['topic']))}")

## 7. Prepare Text for Training

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_text(messages):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(lambda row: {"text": render_text(row["messages"])})
val_dataset = val_dataset.map(lambda row: {"text": render_text(row["messages"])})

# Token length analysis
train_lengths = [len(tokenizer(ex["text"]).input_ids) for ex in train_dataset]
val_lengths = [len(tokenizer(ex["text"]).input_ids) for ex in val_dataset]

length_report = pd.DataFrame({
    "split": ["train", "validation"],
    "count": [len(train_lengths), len(val_lengths)],
    "mean_tokens": [np.mean(train_lengths), np.mean(val_lengths)],
    "max_tokens": [max(train_lengths), max(val_lengths)],
    "p95_tokens": [np.percentile(train_lengths, 95), np.percentile(val_lengths, 95)],
})
length_report

## 8. Fine-Tuning Configuration

### Why LoRA for Translation Style?

- Translation style is a **narrow behavioral objective** — the base model already knows French and English; we are adjusting HOW it translates, not teaching it a new language
- Small adapters can be swapped per domain (regulatory vs. marketing vs. medical) without reloading the full model
- Quick to train and cheap to version

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

MAX_SEQ_LENGTH = int(max(max(train_lengths), max(val_lengths)) + 64)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)

print(f"LoRA rank: {peft_config.r}")
print(f"Max seq: {MAX_SEQ_LENGTH}")
print(f"Epochs: {training_args.num_train_epochs}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=(
        torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else (torch.float16 if torch.cuda.is_available() else torch.float32)
    ),
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("Trainer ready")

In [ ]:
if RUN_TRAINING:
    result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(result)
    print(f"Adapter saved to: {OUTPUT_DIR}")
else:
    print("Training skipped. Set RUN_TRAINING = True to run.")

## 9. Evaluation Framework

### Why Standard Translation Metrics Are Not Enough

BLEU and ROUGE measure n-gram overlap with a reference. They tell you whether the translation uses similar words, but they do NOT measure:

- whether the correct terminology was used
- whether the register is consistent
- whether French typographic conventions were followed
- whether clause structure was preserved

### Evaluation Layers

| Layer | What it measures | Automated? |
|---|---|---|
| **1. BLEU / chrF** | N-gram overlap with reference | Yes |
| **2. Terminology compliance** | Required glossary terms used correctly | Yes |
| **3. Style rubric** | Register, conventions, structure | Semi-auto |
| **4. Human pairwise preference** | Overall quality judgment | No (manual) |

### Layer Weights

For domain-specific translation, terminology compliance matters more than BLEU. A translation with 100% terminology compliance and lower BLEU is usually better than high BLEU with wrong terms.

In [ ]:
import evaluate

bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")


def compute_bleu(predictions, references):
    result = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )
    return round(result["score"], 2)


def compute_rouge(predictions, references):
    result = rouge_metric.compute(
        predictions=predictions,
        references=references,
    )
    return {k: round(v, 4) for k, v in result.items()}


# Test with a reference pair
test_pred = [df.iloc[0]["target"]]
test_ref = [df.iloc[0]["target"]]
print(f"Self-BLEU (should be 100): {compute_bleu(test_pred, test_ref)}")
print(f"Self-ROUGE: {compute_rouge(test_pred, test_ref)}")

In [ ]:
def terminology_score(source, translation):
    source_lower = source.lower()
    translation_lower = translation.lower()
    checked = 0
    matched = 0
    for en_term, fr_term in GLOSSARY.items():
        if en_term in source_lower:
            checked += 1
            fr_normalized = fr_term.replace("\u00e9", "e").replace("\u00e8", "e")
            trans_normalized = translation_lower.replace("\u00e9", "e").replace("\u00e8", "e")
            if fr_normalized in trans_normalized or fr_term in translation_lower:
                matched += 1
    return matched / checked if checked > 0 else 1.0


def style_heuristic_score(translation):
    score = 0.0
    checks = 0

    # Formal register markers
    checks += 1
    informal = ["on doit", "on peut", "on a", "il faut que"]
    if not any(marker in translation.lower() for marker in informal):
        score += 1.0

    # Institutional third person
    checks += 1
    first_person = [" nous ", " notre ", " nos ", "je ", "j'"]
    if not any(marker in translation.lower() for marker in first_person):
        score += 1.0

    # French punctuation (space before colon)
    checks += 1
    if " :" in translation or ":" not in translation:
        score += 1.0

    # No English leakage (untranslated English words)
    checks += 1
    english_leaks = ["shall", "pursuant", "whereas", "thereof", "herein"]
    if not any(word in translation.lower() for word in english_leaks):
        score += 1.0

    return round(score / checks, 3) if checks > 0 else 1.0


# Test
sample_source = df.iloc[0]["source"]
sample_target = df.iloc[0]["target"]
print(f"Terminology score: {terminology_score(sample_source, sample_target):.2f}")
print(f"Style score: {style_heuristic_score(sample_target):.2f}")

In [ ]:
def full_eval(sources, predictions, references, label):
    bleu = compute_bleu(predictions, references)
    rouge = compute_rouge(predictions, references)

    term_scores = [terminology_score(s, p) for s, p in zip(sources, predictions)]
    style_scores = [style_heuristic_score(p) for p in predictions]

    mean_term = np.mean(term_scores)
    mean_style = np.mean(style_scores)

    print(f"\n{'=' * 55}")
    print(f"  {label}")
    print(f"{'=' * 55}")
    print(f"  BLEU:                {bleu:.2f}")
    print(f"  ROUGE-L:             {rouge.get('rougeL', 0):.4f}")
    print(f"  Terminology score:   {mean_term:.1%}")
    print(f"  Style score:         {mean_style:.1%}")
    print(f"\n  Terminology by pair:")
    for i, (s, ts) in enumerate(zip(sources, term_scores)):
        status = "PASS" if ts >= 1.0 else f"PARTIAL ({ts:.0%})"
        print(f"    [{i+1}] {status}  {s[:60]}...")

    return {
        "label": label,
        "bleu": bleu,
        "rouge_l": rouge.get("rougeL", 0),
        "mean_terminology": mean_term,
        "mean_style": mean_style,
        "term_scores": term_scores,
        "style_scores": style_scores,
    }


# Demo: eval reference translations against themselves (perfect score baseline)
demo_sources = list(df["source"])
demo_refs = list(df["target"])
baseline_eval = full_eval(demo_sources, demo_refs, demo_refs, "Reference (self-eval)")

## 10. Generate and Evaluate: Base vs. Fine-Tuned

After training, compare the base model and the fine-tuned model on the validation set using all four evaluation layers.

In [ ]:
from transformers import pipeline as hf_pipeline
from peft import PeftModel


def generate_translation(generator, source_text, max_new_tokens=300):
    messages = [
        {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
        {"role": "user", "content": source_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False, temperature=None)
    generated = output[0]["generated_text"]
    return generated[len(prompt):].strip()


if RUN_GENERATION_EVAL:
    # Base model
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    base_gen = hf_pipeline("text-generation", model=base_model, tokenizer=tokenizer)

    # Fine-tuned model
    ft_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    ft_model = PeftModel.from_pretrained(ft_base, str(OUTPUT_DIR))
    ft_gen = hf_pipeline("text-generation", model=ft_model, tokenizer=tokenizer)

    # Look up references
    pair_lookup = {row["pair_id"]: row for _, row in df.iterrows()}
    val_sources = []
    val_refs = []
    base_preds = []
    ft_preds = []

    for _, row in val_df.iterrows():
        pair = pair_lookup[row["pair_id"]]
        source = pair["source"]
        ref = pair["target"]
        val_sources.append(source)
        val_refs.append(ref)
        base_preds.append(generate_translation(base_gen, source))
        ft_preds.append(generate_translation(ft_gen, source))

    base_results = full_eval(val_sources, base_preds, val_refs, "Base Model")
    ft_results = full_eval(val_sources, ft_preds, val_refs, "Fine-Tuned (LoRA)")
else:
    print("Generation eval skipped. Set RUN_GENERATION_EVAL = True after training.")

## 11. Results Comparison

In [ ]:
if RUN_GENERATION_EVAL:
    comparison = pd.DataFrame([
        {
            "Model": base_results["label"],
            "BLEU": f"{base_results['bleu']:.2f}",
            "ROUGE-L": f"{base_results['rouge_l']:.4f}",
            "Terminology": f"{base_results['mean_terminology']:.0%}",
            "Style": f"{base_results['mean_style']:.0%}",
        },
        {
            "Model": ft_results["label"],
            "BLEU": f"{ft_results['bleu']:.2f}",
            "ROUGE-L": f"{ft_results['rouge_l']:.4f}",
            "Terminology": f"{ft_results['mean_terminology']:.0%}",
            "Style": f"{ft_results['mean_style']:.0%}",
        },
    ])
    print("HEAD-TO-HEAD COMPARISON")
    print("=" * 65)
    print(comparison.to_string(index=False))

    # Side-by-side examples
    print("\n\nSIDE-BY-SIDE EXAMPLES")
    print("=" * 65)
    for i in range(min(3, len(val_sources))):
        print(f"\n--- Example {i+1} ---")
        print(f"Source:     {val_sources[i][:100]}...")
        print(f"Reference:  {val_refs[i][:100]}...")
        print(f"Base:       {base_preds[i][:100]}...")
        print(f"Fine-tuned: {ft_preds[i][:100]}...")
else:
    print("Comparison requires generation eval. Set RUN_GENERATION_EVAL = True.")

## 12. Save Experiment Results

In [ ]:
experiment_log = {
    "timestamp": datetime.now().isoformat(),
    "task": "translation_style_finetuning",
    "direction": "EN -> FR",
    "domain": "financial_regulatory",
    "dataset_size": len(df),
    "train_size": len(train_df),
    "val_size": len(val_df),
    "base_model": BASE_MODEL,
    "style_policy": STYLE_POLICY,
    "glossary": GLOSSARY,
}

log_path = ARTIFACT_DIR / "translation_experiment_log.json"
log_path.write_text(json.dumps(experiment_log, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"Saved: {log_path}")

## 13. Limitations and Failure Modes

### Fundamental Limitations

| Limitation | Why | Mitigation |
|---|---|---|
| **Terminology drift** | The model may use the right term 90% of the time but slip on long documents | Post-processing glossary enforcement or constrained decoding |
| **Register inconsistency** | Formal register can decay in long outputs, especially in informal-looking source text | Train on full-paragraph examples, not just sentences |
| **Hallucinated content** | The model may add information not in the source (common in regulatory translation) | Length-ratio checks + human review for high-stakes documents |
| **New terminology** | Terms not in the training glossary will be translated generically | Maintain and update the glossary; retrain periodically |
| **Ambiguous source** | When the English is ambiguous, the model may pick incorrectly | Flag ambiguous segments for human review |
| **Cultural/legal adaptation** | Some concepts need adaptation, not translation (e.g., common law → civil law) | These require expert human translators; fine-tuning cannot solve this |

### Translation-Specific Failure Modes

**1. The Consistency Problem**

A fine-tuned model may translate "investment firm" as "entreprise d'investissement" in one paragraph and "societe d'investissement" in the next. This is worse than consistently using the wrong term because it suggests unreliability.

**Mitigation:** Post-processing that enforces glossary terms, or translation memory integration.

**2. The Overcorrection Problem**

After fine-tuning on highly formal regulatory text, the model may produce overly stiff translations even when the source text is less formal (e.g., translating a casual explanatory note in the same register as a legal article).

**Mitigation:** Include a few examples of less formal source text with appropriately less formal (but still professional) translations.

**3. The Reference Bias Problem**

BLEU and ROUGE reward translations that match the reference wording. But there are often multiple correct translations of the same sentence. An alternative translation that is equally valid but differently worded will score lower.

**Mitigation:** Use multiple reference translations when available. Weight terminology compliance and style rubric higher than BLEU.

**4. The Dataset Scale Problem**

20 pairs (as in this demo) are not enough for production fine-tuning. Expected minimums:

| Use case | Minimum pairs | Why |
|---|---|---|
| Terminology calibration | 200-500 | Need exposure to every glossary term in context |
| Style transfer | 500-1,000 | Register consistency requires many examples |
| Full domain adaptation | 1,000-5,000 | Coverage of all regulatory sub-topics |

**5. The Evaluation Gap**

Automated metrics (BLEU, terminology, style heuristics) capture only part of translation quality. The remaining gap — fluency, naturalness, legal accuracy — requires human evaluation by bilingual domain experts.

### When NOT to Fine-Tune for Translation

| Scenario | Better approach |
|---|---|
| Terminology is the only issue | Constrained decoding or post-processing glossary replacement |
| Source text varies wildly in domain | Use RAG to inject domain-specific glossary at inference time |
| You have fewer than 100 quality pairs | Few-shot prompting with glossary in the prompt |
| The regulatory framework changes frequently | Glossary updates + prompting; avoid baking volatile terms into weights |
| You need legal-grade accuracy | Human translation with AI as a first draft |

## 14. Key Takeaways

### What We Built

- A **domain-specific translation fine-tuning pipeline** targeting financial regulatory EN→FR translation
- A **glossary-aware dataset curation process** with automated quality checks
- A **multi-layer evaluation framework** combining BLEU, terminology compliance, and style heuristics
- Clear documentation of **limitations and failure modes** specific to translation fine-tuning

### Lessons for Translation Style Fine-Tuning

1. **Define the terminology glossary before building the dataset.** The glossary is the contract between your training data and your evaluation.

2. **Validate every training pair against the glossary.** If you train on inconsistent terminology, the model learns inconsistency.

3. **BLEU is necessary but not sufficient.** Terminology compliance and style adherence matter more for domain translation. A translation that uses the wrong regulatory term is worse than one with lower BLEU but correct terms.

4. **Include edge cases in training.** Abbreviations, cross-references, nested clauses, and mixed-formality source text are where models fail.

5. **Plan for glossary updates.** Regulatory terminology evolves. Your retraining pipeline must accommodate new terms without losing performance on existing ones.

6. **Human evaluation is not optional.** Automated metrics cannot assess legal accuracy, naturalness, or cultural appropriateness. Budget for bilingual expert review.

### Production Checklist

- [ ] Glossary defined and versioned alongside the model
- [ ] Training pairs validated against glossary (100% compliance)
- [ ] Evaluation includes terminology compliance, not just BLEU
- [ ] Edge-case test slices (abbreviations, cross-references, long clauses)
- [ ] Human review on a sample of model outputs before deployment
- [ ] Regression testing when the glossary or training data is updated
- [ ] Post-processing glossary enforcement as a safety net